# Spread Exploration and Cointegration Screening

This notebook screens 16 candidate equity pairs across 8 sectors for cointegration over the in-sample period (January 2018 -- December 2023) using the Engle--Granger two-step procedure. Daily closing prices are retrieved from LSEG Workspace and split into in-sample and out-of-sample sets. For each cointegrated pair, the log-price spread is characterised by its half-life of mean reversion, Hurst exponent, rolling z-score, and distribution.

**Pipeline overview.**

| Step | Function | Description |
|------|----------|-------------|
| 1 | `get_close_prices()` | Retrieve daily closing prices for all 26 tickers; split IS / OOS |
| 2 | `screen_pairs()` | Run Engle--Granger tests on all 16 candidate pairs at 5% significance |
| 3 | `spread_summary()` | Compute half-life, Hurst exponent, ADF statistic, and z-score for each cointegrated spread |
| 4 | Visualise | Normalised log prices, spread levels with $\sigma$-bands, rolling half-life, rolling z-score, and spread distributions |
| 5 | Save outputs | Write cointegrated pairs, spread summaries, and log prices to `data/processed/` |

**Cointegration test.** `screen_pairs()` applies OLS regression of $\ln P^y$ on $\ln P^x$ (Step 1) and an Augmented Dickey--Fuller test on the residuals (Step 2), using MacKinnon (1991) critical values. A pair is flagged as cointegrated if the ADF $p$-value is below 0.05, both constituent series are $I(1)$, and the spread is stationary.

**Spread construction.** The log-price spread is defined as:

$$S_t = \ln P^y_t - \hat{\beta}\,\ln P^x_t - \hat{\alpha}$$

where $\hat{\beta}$ is the OLS hedge ratio and $\hat{\alpha}$ is the intercept. If the pair is cointegrated, $S_t$ is stationary with mean zero by construction, and deviations from zero represent temporary mispricings expected to revert.

**Outputs.** Confirmed cointegrated pairs (with hedge ratios and intercepts), spread summary statistics, and log prices are saved to `data/processed/` for use in `02_cointegration_validation.ipynb` and `03_return_estimation.ipynb`.

# Imports

In [1]:
import sys
import os
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data import get_close_prices, get_market_cap, get_ohlcv
from src.modelling import TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END, INTERVAL, TICKER_NAMES, CANDIDATE_PAIRS
from src.modelling import engle_granger_test, screen_pairs, adf_test # co-integration
from src.modelling import spread_summary, compute_spread, compute_zscore, compute_rolling_half_life, compute_rolling_zscore # spread analysis


# Getting Historical Prices

For cointegration-based spread analysis, 16 candidate pairs across 8 sectors are studied. Same-sector pairs share common macroeconomic drivers --- such as interest rates, commodity prices, and regulation --- making them strong candidates for a stable long-run equilibrium. Market caps are as of 29 December 2023 (end of in-sample period), sorted descending by average pair market cap:

| Sector           | Pair                       | Tickers          | Mkt Cap (Y) | Mkt Cap (X) | Avg     |
| ---------------- | -------------------------- | ---------------- | ----------: | ----------: | ------: |
| Cloud            | Amazon / Microsoft         | AMZN.O / MSFT.O  | $1,570B     | $2,795B     | $2,182B |
| Tech Mega-caps   | Amazon / Alphabet          | AMZN.O / GOOGL.O | $1,570B     | $1,755B     | $1,663B |
| Digital Ads      | Meta / Alphabet            | META.O / GOOGL.O | $910B       | $1,755B     | $1,333B |
| Semiconductors   | Nvidia / TSMC              | NVDA.O / TSM.N   | $1,223B     | $501B       | $862B   |
| Semiconductors   | Nvidia / AMD               | NVDA.O / AMD.O   | $1,223B     | $238B       | $731B   |
| Banking          | JPMorgan / BofA            | JPM.N / BAC.N    | $492B       | $266B       | $379B   |
| Semiconductors   | TSMC / Intel               | TSM.N / INTC.O   | $501B       | $212B       | $357B   |
| Oil Majors       | ExxonMobil / Chevron       | XOM.N / CVX.N    | $400B       | $281B       | $340B   |
| Pharma           | Merck / AbbVie             | MRK.N / ABBV.N   | $276B       | $274B       | $275B   |
| Pharma           | J&J / Pfizer               | JNJ.N / PFE.N    | $377B       | $163B       | $270B   |
| Beverages        | Coca-Cola / PepsiCo        | KO.N / PEP.O     | $255B       | $234B       | $244B   |
| Semiconductors   | AMD / Intel                | AMD.O / INTC.O   | $238B       | $212B       | $225B   |
| Retail           | Costco / Target            | COST.O / TGT.N   | $293B       | $66B        | $179B   |
| Investment Banks | Goldman / Morgan Stanley   | GS.N / MS.N      | $126B       | $153B       | $140B   |
| Oil Services     | Schlumberger / Halliburton | SLB.N / HAL.N    | $74B        | $32B        | $53B    |
| Airlines         | Delta / United             | DAL.N / UAL.O    | $26B        | $14B        | $20B    |

Daily closing prices are retrieved for all tickers over the full date range and split into in-sample (training) and out-of-sample (test) sets:

- **Training period**: 1 January 2018 -- 31 December 2023
- **Test period**: 1 January 2024 -- 31 December 2025

## Get Market Cap
Obtain market cap prices as of 29th December 2023 (last trading day of 2023 - end of test period)

In [2]:
market_cap_df = get_market_cap(
    TICKERS,
    start="2023-12-29",
    end="2023-12-29",
)

market_cap_df

[cache hit] ABBV-N_AMD-O_AMZN-O_BAC-N_COST-O_CVX-N_D__1D__c9963c7065.csv


,NVDA.O,AMD.O,TSM.N,INTC.O,KO.N,PEP.O,COST.O,TGT.N,JPM.N,BAC.N,...,DAL.N,UAL.O,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N,MRK.N,ABBV.N
Date,,,,,,,,,,,,,,,,,,,,,
2023-12-29,1223193400000,2.381407e+11,5.011967e+11,211854000000,2.547788e+11,2.335069e+11,2.928963e+11,6.574987e+10,4.917605e+11,2.664554e+11,...,2.588653e+10,1.353398e+10,1.570153e+12,2.794828e+12,9.096286e+11,1755459040000,3.773169e+11,1.625602e+11,2.762592e+11,2.736053e+11


## Obtain ticker prices

Obtain ticker prices for all TICKER symbols from start of training period (2019-01-01) to end of training period (2023-12-31) date with interval of 1 day

In [3]:
prices_df = get_close_prices(
    TICKERS,
    start=TRAIN_START,
    end=TRAIN_END,
    interval=INTERVAL
)

[cache partial] Fetching 2018-01-01 → 2018-01-01
LSEG session opened.
[cache partial] Skipping (no trading data): LSEG returned no data for ['NVDA.O', 'AMD.O', 'TSM.N', 'INTC.O', 'KO.N', 'PEP.O', 'COST.O', 'TGT.N', 'JPM.N', 'BAC.N', 'GS.N', 'MS.N', 'XOM.N', 'CVX.N', 'SLB.N', 'HAL.N', 'DAL.N', 'UAL.O', 'AMZN.O', 'MSFT.O', 'META.O', 'GOOGL.O', 'JNJ.N', 'PFE.N', 'MRK.N', 'ABBV.N'] (2018-01-01 to 2018-01-01)


In [4]:
print(prices_df.shape)
prices_df.head()

(1509, 26)


,NVDA.O,AMD.O,TSM.N,INTC.O,KO.N,PEP.O,COST.O,TGT.N,JPM.N,BAC.N,...,DAL.N,UAL.O,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N,MRK.N,ABBV.N
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,4.98375,10.98,41.02,46.85,45.54,118.06,179.432718,67.63,107.95,29.90,...,56.74,68.94,59.4505,85.95,181.42,53.6605,139.23,34.543262,53.607232,98.41
2018-01-03,5.31175,11.55,41.71,45.26,45.44,117.75,181.586063,67.17,108.06,29.80,...,55.69,68.49,60.2100,86.35,184.67,54.5760,140.56,34.799208,53.530950,99.95
2018-01-04,5.33975,12.12,41.49,44.43,46.08,118.33,180.175907,65.85,109.04,30.19,...,55.69,69.26,60.4795,87.11,184.33,54.7880,140.55,34.875044,54.398658,99.38
2018-01-05,5.38500,11.88,42.46,44.74,46.07,118.67,178.889617,66.55,108.34,30.33,...,55.97,69.36,61.4570,88.19,186.85,55.5145,141.71,34.941400,54.341447,101.11
2018-01-08,5.55000,12.28,42.44,44.74,46.00,117.99,179.585167,67.18,108.50,30.12,...,54.68,68.51,62.3435,88.28,188.28,55.7105,141.89,34.552741,54.026783,99.49


In [5]:
prices_df.to_csv('../../data/processed/is_prices_df.csv')

### Testing it works

Plotting the closing price of NVDA.O from January 1, 2018 to December 31, 2023

In [6]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prices_df.index,
    y=prices_df['NVDA.O'],
    mode='lines',
    name='NVDA.O',
    line=dict(color='#76b900', width=1.5)
))

fig.update_layout(
    title='NVDA.O Close Price (Jan 2019 – Dec 2023)',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    xaxis_rangeslider_visible=False,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

# Spread Analysis

## Obtain Pre-Selected Candidate Pairs

In [7]:
CANDIDATE_PAIRS

[('NVDA.O', 'AMD.O'),
 ('NVDA.O', 'TSM.N'),
 ('AMD.O', 'INTC.O'),
 ('TSM.N', 'INTC.O'),
 ('KO.N', 'PEP.O'),
 ('COST.O', 'TGT.N'),
 ('JPM.N', 'BAC.N'),
 ('GS.N', 'MS.N'),
 ('XOM.N', 'CVX.N'),
 ('SLB.N', 'HAL.N'),
 ('DAL.N', 'UAL.O'),
 ('AMZN.O', 'MSFT.O'),
 ('META.O', 'GOOGL.O'),
 ('AMZN.O', 'GOOGL.O'),
 ('JNJ.N', 'PFE.N'),
 ('MRK.N', 'ABBV.N')]

## Run Engle-Granger Cointegration Tests on Candidate Pairs

Run the Engle--Granger test on all 16 candidate pairs using log prices. Log prices are used to stabilise variance across tickers with very different price levels, and to ensure the spread $S_t = \ln P^y_t - \hat{\beta}\ln P^x_t - \hat{\alpha}$ is dimensionally consistent.

`screen_pairs()` calls `statsmodels.tsa.stattools.coint` internally, which runs OLS to obtain $\hat{\beta}$ and $\hat{\alpha}$, then applies an ADF unit-root test on the residuals. A pair is flagged as cointegrated if: (1) $p < 0.05$, (2) both series are $I(1)$, and (3) the spread passes the ADF stationarity test.

In [8]:
prices_log = np.log(prices_df)
results_df = screen_pairs(prices_log, CANDIDATE_PAIRS, significance=0.05)

The results DataFrame contains the following columns for each candidate pair:

| Column | Description |
|--------|-------------|
| `hedge_ratio` ($\hat{\beta}$) | OLS coefficient from regressing $\ln P^y$ on $\ln P^x$; the long-term equilibrium ratio |
| `intercept` ($\hat{\alpha}$) | OLS constant; captures the average log-price level difference |
| `adf_stat` | ADF test statistic on the spread residuals |
| `p_value` | MacKinnon (1991) $p$-value; pairs with $p < 0.05$ are flagged as cointegrated |
| `y_is_I1`, `x_is_I1` | Whether each series is integrated of order 1 |
| `is_cointegrated` | True if $p < 0.05$ and both series are $I(1)$ |

Both `hedge_ratio` and `intercept` are required for spread construction in subsequent notebooks.

In [9]:
display(results_df)

,y,x,hedge_ratio,intercept,adf_stat,p_value,y_is_I1,x_is_I1,is_cointegrated
0,GS.N,MS.N,0.780150,2.350976,-3.666910,0.020199,True,True,True
1,KO.N,PEP.O,0.641464,0.788960,-3.470124,0.035137,True,True,True
2,DAL.N,UAL.O,0.671613,1.067783,-3.460167,0.036094,True,True,True
3,CVX.N,XOM.N,0.698093,1.812827,-3.234418,0.064358,True,True,False
4,AMZN.O,GOOGL.O,0.630327,1.942272,-2.441875,0.305287,True,True,False
5,HAL.N,SLB.N,0.997410,-0.344470,-2.369711,0.339331,True,True,False
6,AMD.O,NVDA.O,0.811076,1.999131,-2.310437,0.368357,True,True,False
7,AMZN.O,MSFT.O,0.515737,2.026518,-2.140596,0.455174,True,True,False
8,MRK.N,ABBV.N,0.430569,2.365703,-1.979701,0.539112,True,True,False
9,INTC.O,AMD.O,-0.080956,4.154386,-1.958442,0.550101,True,True,False


### Filter Results

Filter to confirmed cointegrated pairs ($p < 0.05$). These three pairs are passed to all downstream notebooks.

In [10]:
cointegrated_pairs = results_df[results_df['p_value'] < 0.05].copy()
print(cointegrated_pairs[['y', 'x', 'hedge_ratio', 'p_value']])


       y      x  hedge_ratio   p_value
0   GS.N   MS.N     0.780150  0.020199
1   KO.N  PEP.O     0.641464  0.035137
2  DAL.N  UAL.O     0.671613  0.036094


### Compute spreads

The spread $S_t$ between two log-price series is defined as the residual from the OLS regression in Step 1 of the Engle-Granger test:

$$S_t = \ln P^y_t - \hat{\beta} \ln P^x_t - \hat{\alpha}$$

where:
- $\ln P^y_t$ and $\ln P^x_t$ are the log prices of the dependent and independent assets at time $t$
- $\hat{\beta}$ is the hedge ratio — the OLS coefficient from regressing $\ln P^y$ on $\ln P^x$
- $\hat{\alpha}$ is the intercept, capturing the average log-price level difference between the two series

The hedge ratio and intercepts were obtained from the `screen_pairs` function.

If the pair is cointegrated, $S_t$ is stationary with mean zero by construction. Deviations of $S_t$ from zero represent temporary mispricings that the strategy expects to revert.

In [11]:
summaries = []
for _, row in cointegrated_pairs.iterrows():
    y = prices_log[row['y']]
    x = prices_log[row['x']]
    
    summary = spread_summary(
        y, x,
        hedge_ratio=row['hedge_ratio'],
        intercept=row['intercept'],
        window=60
    )
    summary['pair'] = f"{row['y']}/{row['x']}"
    summaries.append(summary)

summary_df = pd.DataFrame(summaries).set_index('pair')
summary_df


,half_life,hurst,current_zscore,spread_mean,spread_std,adf_stat,adf_pvalue,is_stationary
pair,,,,,,,,
GS.N/MS.N,41.773021,0.402840,0.675478,-5.768157e-16,0.055858,-3.665527,0.004625,True
KO.N/PEP.O,40.323210,0.421018,0.808854,-5.673394e-15,0.046996,-3.468984,0.008820,True
DAL.N/UAL.O,52.863642,0.413517,1.314954,7.682243e-15,0.076182,-3.459317,0.009095,True


All three spreads have Hurst exponents below 0.5, confirming mean-reverting dynamics, and half-lives between 41 and 53 days --- a practical timescale consistent with weekly rebalancing. The ADF $p$-values on the spreads (0.005 -- 0.009) are substantially more significant than the pair-level Engle--Granger $p$-values (0.020 -- 0.036), reflecting the power gain from the two-step procedure. These properties satisfy the statistical preconditions for fitting an Ornstein--Uhlenbeck model to each spread in `03_return_estimation.ipynb`.

## Visualise Spreads

### Visualise co-movements

In [12]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=len(cointegrated_pairs), cols=1, subplot_titles=[
    f"{row['y']} vs {row['x']} — Normalised Log Prices"
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    y = prices_df[row['y']]
    x = prices_df[row['x']]
    
    # Normalise to 0 at start
    y_norm = np.log(y) - np.log(y.iloc[0])
    x_norm = np.log(x) - np.log(x.iloc[0])

    fig.add_trace(go.Scatter(
        x=y_norm.index, y=y_norm.values,
        name=row['y'], line=dict(width=1.5)
    ), row=i, col=1)
    
    fig.add_trace(go.Scatter(
        x=x_norm.index, y=x_norm.values,
        name=row['x'], line=dict(width=1.5, dash='dash')
    ), row=i, col=1)

fig.update_layout(
    height=900,
    title_text="Normalised Log Price Series — Cointegrated Pairs (Jan 2019 – Dec 2023)",
    showlegend=True
)
fig.show()

#### Load to matplotlib

In [13]:
prices_log = np.log(prices_df)

In [61]:
import matplotlib.dates as mdates
import scienceplots

plt.style.use(['science', 'high-contrast', 'ieee'])
plt.rcParams['text.usetex'] = True
plt.rcParams.update({
    'font.size':        8,
    'axes.labelsize':   8,
    'axes.titlesize':   8,
    'xtick.labelsize':  7,
    'ytick.labelsize':  7,
    'legend.fontsize':  7,
    'font.family': 'serif',
    'font.serif' : ['Computer Modern Roman'],
})

fig, axes = plt.subplots(
    nrows=len(cointegrated_pairs),
    ncols=1,
    figsize=(5.9, 5.90),
    sharex=True
)
if len(cointegrated_pairs) == 1:
    axes = [axes]

panel_labels = [f"({chr(97 + i)})" for i in range(len(cointegrated_pairs))]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    y = prices_log[row["y"]]
    x = prices_log[row["x"]]
    y_norm = y - y.iloc[0]
    x_norm = x - x.iloc[0]
    y_name = TICKER_NAMES.get(row["y"], row["y"])
    x_name = TICKER_NAMES.get(row["x"], row["x"])

    ax.plot(y_norm.index, y_norm.values, label=y_name, linewidth=1.5)
    ax.plot(x_norm.index, x_norm.values, label=x_name, linestyle="--", linewidth=1.5)
    ax.set_title(f"{label} {y_name}/{x_name}", loc="left", fontsize=9, fontweight="bold")
    ax.legend(loc="upper right", ncol=2, fontsize=7, frameon=True)

fig.supylabel("Log price (normalised)", x=0.02, fontsize=10)
axes[-1].set_xlabel("Date", fontsize=10)
axes[-1].tick_params(axis='x', labelsize=9)
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.tight_layout(rect=[0, 0.02, 1, 1])
fig.savefig("figures/01_normalised_prices.pdf", bbox_inches="tight")
plt.close(fig)

### Spread Characterisation

In [47]:
from plotly.subplots import make_subplots

pairs      = summary_df.index.tolist()
half_lives = summary_df['half_life'].tolist()
hursts     = summary_df['hurst'].tolist()
zscores    = summary_df['current_zscore'].tolist()
adf_pvals  = summary_df['adf_pvalue'].tolist()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Half-Life of Mean Reversion (Trading Days)",
        "Hurst Exponent",
        "Current Z-Score (End of Training Window)",
        "ADF p-value on Spread"
    )
)

# --- Half-life bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=half_lives,
    marker_color=['green' if h < 60 else 'orange' for h in half_lives],
    text=[f"{h:.1f}d" for h in half_lives], textposition='auto',
    name='Half-Life'
), row=1, col=1)
fig.add_hline(y=60, line=dict(color='red', dash='dash'), annotation_text='60d threshold', row=1, col=1)

# --- Hurst exponent bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=hursts,
    marker_color=['green' if h < 0.5 else 'red' for h in hursts],
    text=[f"{h:.3f}" for h in hursts], textposition='auto',
    name='Hurst'
), row=1, col=2)
fig.add_hline(y=0.5, line=dict(color='red',    dash='dash'), annotation_text='Random walk (0.5)', row=1, col=2)

# --- Current z-score bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=zscores,
    marker_color=['red' if abs(z) > 2 else 'steelblue' for z in zscores],
    text=[f"{z:.2f}" for z in zscores], textposition='auto',
    name='Z-Score'
), row=2, col=1)
fig.add_hline(y=2,  line=dict(color='orange', dash='dot'), row=2, col=1)
fig.add_hline(y=-2, line=dict(color='orange', dash='dot'), row=2, col=1)
fig.add_hline(y=0,  line=dict(color='gray',   dash='dash'), row=2, col=1)

# --- ADF p-value bar chart ---
fig.add_trace(go.Bar(
    x=pairs, y=adf_pvals,
    marker_color=['green' if p < 0.05 else 'red' for p in adf_pvals],
    text=[f"{p:.3f}" for p in adf_pvals], textposition='auto',
    name='ADF p-value'
), row=2, col=2)
fig.add_hline(y=0.05, line=dict(color='red', dash='dash'), annotation_text='5% significance', row=2, col=2)

fig.update_layout(
    height=700,
    title_text="Spread Characterisation Summary — Cointegrated Pairs",
    showlegend=False
)

fig.show()

#### Save to Matplotlib

In [63]:
import matplotlib.patches as mpatches

pairs      = summary_df.index.tolist()
half_lives = summary_df["half_life"].tolist()
hursts     = summary_df["hurst"].tolist()
zscores    = summary_df["current_zscore"].tolist()
adf_pvals  = summary_df["adf_pvalue"].tolist()

PASS  = dict(color="white", edgecolor="black", linewidth=0.8)
FAIL  = dict(color="white", edgecolor="black", linewidth=0.8, hatch="///")
REF   = dict(color="black", linestyle="--", linewidth=1)

fig, axes = plt.subplots(2, 2, figsize=(5.9, 5.0))
ax_hl, ax_hu, ax_z, ax_adf = axes.flatten()

# Half-life
for x, (pair, h) in enumerate(zip(pairs, half_lives)):
    ax_hl.bar(x, h, **(PASS if h < 60 else FAIL))
    ax_hl.text(x, h, f"{h:.1f}d", ha="center", va="bottom", fontsize=9)
ax_hl.set_xticks(range(len(pairs))); ax_hl.set_xticklabels(pairs, fontsize=7)
ax_hl.axhline(60, **REF)
ax_hl.text(len(pairs)-1, 60, "60d threshold", ha="right", va="bottom")
ax_hl.set_ylabel("Days",fontsize=10)
ax_hl.set_ylim(0,65)
# Hurst
for x, (pair, h) in enumerate(zip(pairs, hursts)):
    ax_hu.bar(x, h, **(PASS if h < 0.5 else FAIL))
    ax_hu.text(x, h, f"{h:.3f}", ha="center", va="bottom", fontsize=9)
ax_hu.set_xticks(range(len(pairs))); ax_hu.set_xticklabels(pairs, fontsize=7)
ax_hu.axhline(0.5, **REF)
ax_hu.text(len(pairs)-1, 0.5, "Random walk (0.5)", ha="right", va="bottom")
ax_hu.set_ylabel("H",fontsize=10)
ax_hu.set_ylim(0,0.55)

# Z-score
for x, (pair, z) in enumerate(zip(pairs, zscores)):
    ax_z.bar(x, z, **(FAIL if abs(z) > 2 else PASS))
    ax_z.text(x, z, f"{z:.2f}", ha="center", va=("bottom" if z >= 0 else "top"), fontsize=9)
ax_z.set_xticks(range(len(pairs))); ax_z.set_xticklabels(pairs, fontsize=7)
ax_z.axhline(0,  color="black", linestyle="--", linewidth=1)
ax_z.axhline( 2, color="black", linestyle=":",  linewidth=1)
ax_z.axhline(-2, color="black", linestyle=":",  linewidth=1)
ax_z.set_ylabel("Z-score", fontsize=10)

# ADF p-value
for x, (pair, p) in enumerate(zip(pairs, adf_pvals)):
    ax_adf.bar(x, p, **(PASS if p < 0.05 else FAIL))
    ax_adf.text(x, p, f"{p:.3f}", ha="center", va="bottom", fontsize=9)
ax_adf.set_xticks(range(len(pairs))); ax_adf.set_xticklabels(pairs, fontsize=7)
ax_adf.axhline(0.05, **REF)
ax_adf.text(len(pairs)-1, 0.05, r"$5\%$ significance", ha="right", va="bottom")
ax_adf.set_ylabel("p-value", fontsize=10)
ax_adf.set_ylim(0, 0.055)

# Panel labels (a)–(d) instead of titles
for ax, label in zip(axes.flatten(), ["(a)", "(b)", "(c)", "(d)"]):
    ax.set_title(label, loc="left", fontsize=10, fontweight="bold")

# Shared legend
pass_patch = mpatches.Patch(facecolor="white", edgecolor="black", label="Pass threshold")
fail_patch = mpatches.Patch(facecolor="white", edgecolor="black", hatch="///", label="Fail threshold")
fig.legend(handles=[pass_patch, fail_patch], loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02), fontsize=9)

# no rotation needed for short panel labels

fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/01_spread_characterisation.pdf", bbox_inches="tight")

plt.close(fig)

### Spread level between +1 std to +2 std

In [49]:
fig = make_subplots(rows=3, cols=1, subplot_titles=[
    f"{row['y']}/{row['x']} — Spread Level" 
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']], 
                           row['hedge_ratio'], row['intercept'])
    
    # Spread line
    fig.add_trace(go.Scatter(
        x=spread.index, y=spread.values, 
        line=dict(width=1, color='steelblue'), 
        name='Spread'), row=i, col=1)

    # Mean
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean()] * len(spread),
        line=dict(color='red', dash='dash', width=1),
        name='Mean', showlegend=(i == 1)), row=i, col=1)

    # ±1σ
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() + spread.std()] * len(spread),
        line=dict(color='orange', dash='dot', width=1),
        name='+1σ', showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() - spread.std()] * len(spread),
        line=dict(color='orange', dash='dot', width=1),
        name='-1σ', showlegend=(i == 1)), row=i, col=1)

    # ±2σ
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() + 2*spread.std()] * len(spread),
        line=dict(color='red', dash='dot', width=1),
        name='+2σ', showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(
        x=spread.index, y=[spread.mean() - 2*spread.std()] * len(spread),
        line=dict(color='red', dash='dot', width=1),
        name='-2σ', showlegend=(i == 1)), row=i, col=1)

fig.update_layout(
    height=900, 
    title_text="Spread Levels with σ Bands", 
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

#### Save as matplotlib

In [50]:
import matplotlib.dates as mdates
import scienceplots
plt.style.use(['science', 'high-contrast', 'ieee'])
plt.rcParams['text.usetex'] = True
plt.rcParams.update({
    'font.size':        8,
    'axes.labelsize':   8,
    'axes.titlesize':   8,
    'xtick.labelsize':  7,
    'ytick.labelsize':  7,
    'legend.fontsize':  7,
})
n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(5.9, 5.31), sharex=True)
if n_pairs == 1:
    axes = [axes]
panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]
for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = prices_log[row["y"]] - prices_log[row["x"]]
    mu    = spread.mean()
    sigma = spread.std(ddof=1)
    first = (label == panel_labels[0])
    pair_name = f"{row['y']}/{row['x']}"
    ax.plot(spread.index, spread.values, linewidth=1, label="Spread" if first else None)
    ax.axhline(mu,           linestyle="--", linewidth=1,   label="Mean" if first else None)
    ax.axhline(mu + sigma,   linestyle=":",  linewidth=0.8, label=r"$\pm1\sigma$" if first else None)
    ax.axhline(mu - sigma,   linestyle=":",  linewidth=0.8)
    ax.axhline(mu + 2*sigma, linestyle="-.", linewidth=0.8, label=r"$\pm2\sigma$" if first else None)
    ax.axhline(mu - 2*sigma, linestyle="-.", linewidth=0.8)
    ax.set_title(f"{label} {pair_name}", loc="left", fontsize=10, fontweight="bold")
    ax.set_ylabel("Spread")
    # Annotate reference lines on the right
    ax.annotate(r"$\mu$",         xy=(1, mu),           xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$+1\sigma$",    xy=(1, mu + sigma),   xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$-1\sigma$",    xy=(1, mu - sigma),   xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$+2\sigma$",    xy=(1, mu + 2*sigma), xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$-2\sigma$",    xy=(1, mu - 2*sigma), xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
axes[-1].set_xlabel("Date", fontsize=10)
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/01_spread_levels.pdf", bbox_inches="tight")
plt.close(fig)

### Z-scores visualisation

In [51]:
fig = make_subplots(rows=3, cols=1, subplot_titles=[
    f"{row['y']}/{row['x']} — Rolling Z-Score (60d)"
    for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']],
                           row['hedge_ratio'], row['intercept'])
    zscore = compute_zscore(spread, window=60)

    fig.add_trace(go.Scatter(x=zscore.index, y=zscore.values,
                             line=dict(width=1, color='purple')), row=i, col=1)
    fig.add_hline(y=0,  line=dict(color='red',    dash='dash'), row=i, col=1)
    fig.add_hline(y=2,  line=dict(color='orange', dash='dot'),  row=i, col=1)
    fig.add_hline(y=-2, line=dict(color='orange', dash='dot'),  row=i, col=1)

fig.update_layout(height=900, title_text="Rolling Z-Scores", showlegend=False)
fig.show()


#### Save to Matplotlib

In [52]:
import matplotlib.dates as mdates
import scienceplots
plt.style.use(['science', 'high-contrast', 'ieee'])
plt.rcParams['text.usetex'] = True
plt.rcParams.update({
    'font.size':        8,
    'axes.labelsize':   8,
    'axes.titlesize':   8,
    'xtick.labelsize':  7,
    'ytick.labelsize':  7,
    'legend.fontsize':  7,
})
window = 60
n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(5.9, 5.31), sharex=True)
if n_pairs == 1:
    axes = [axes]
panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]
for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = compute_spread(
        prices_log[row["y"]],
        prices_log[row["x"]],
        row["hedge_ratio"],
        row["intercept"],
    ).dropna()
    zscore = compute_zscore(spread, window=window).dropna()
    first = (label == panel_labels[0])
    pair_name = f"{row['y']}/{row['x']}"
    ax.plot(zscore.index, zscore.values, linewidth=1, label="Z-score" if first else None)
    ax.axhline(0,  linestyle="--", linewidth=1,   label="Mean" if first else None)
    ax.axhline( 2, linestyle=":",  linewidth=0.8, label=r"$\pm2\sigma$" if first else None)
    ax.axhline(-2, linestyle=":",  linewidth=0.8)
    ax.set_title(f"{label} {pair_name}", loc="left", fontsize=10, fontweight="bold")
    ax.set_ylabel("Z-score")
    ax.annotate(r"$+2\sigma$", xy=(1, 2),  xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$-2\sigma$", xy=(1, -2), xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$\mu$",      xy=(1, 0),  xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/01_spread_zscores.pdf", bbox_inches="tight")
plt.close(fig)


### Spread Distribution

In [53]:
from scipy import stats

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    f"{row['y']}/{row['x']}" for _, row in cointegrated_pairs.iterrows()
])

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(prices_log[row['y']], prices_log[row['x']],
                           row['hedge_ratio'], row['intercept']).dropna()

    # Histogram
    fig.add_trace(go.Histogram(x=spread.values, nbinsx=50, 
                               histnorm='probability density',
                               marker_color='steelblue', opacity=0.7,
                               name='Spread'), row=1, col=i)

    # Normal overlay
    x_range = np.linspace(spread.min(), spread.max(), 200)
    normal_pdf = stats.norm.pdf(x_range, spread.mean(), spread.std())
    fig.add_trace(go.Scatter(x=x_range, y=normal_pdf,
                             line=dict(color='red', width=2),
                             name='Normal fit'), row=1, col=i)

fig.update_layout(height=400, title_text="Spread Distributions", showlegend=False)
fig.show()


#### Save to Matplotlib

In [64]:
from scipy import stats
import scienceplots
plt.style.use(['science', 'high-contrast', 'ieee'])
plt.rcParams['text.usetex'] = True
plt.rcParams.update({
    'font.size':        8,
    'axes.labelsize':   8,
    'axes.titlesize':   8,
    'xtick.labelsize':  7,
    'ytick.labelsize':  7,
    'legend.fontsize':  7,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],  # LaTeX default font
})
n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=1, ncols=n_pairs, figsize=(5.9, 2.20), sharey=True)
if n_pairs == 1:
    axes = [axes]
panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]
for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = compute_spread(
        prices_log[row["y"]],
        prices_log[row["x"]],
        row["hedge_ratio"],
        row["intercept"],
    ).dropna()
    x_range    = np.linspace(spread.min(), spread.max(), 200)
    normal_pdf = stats.norm.pdf(x_range, spread.mean(), spread.std())
    first = (label == panel_labels[0])
    pair_name = f"{row['y']}/{row['x']}"
    ax.hist(spread.values, bins=50, density=True, alpha=0.6,
            label="Spread" if first else None)
    ax.plot(x_range, normal_pdf, linewidth=1.5,
            label="Normal fit" if first else None)
    ax.set_title(f"{label} {pair_name}", loc="left", fontsize=10, fontweight="bold")
    ax.set_xlabel("Spread")
axes[0].set_ylabel("Density")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig("figures/01_spread_distributions.pdf", bbox_inches="tight")
plt.close(fig)


### Rolling Half-Life

In [55]:
fig = make_subplots(
    rows=len(cointegrated_pairs), cols=1,
    subplot_titles=[
        f"{row['y']}/{row['x']} Rolling Half-Life"
        for _, row in cointegrated_pairs.iterrows()
    ],
    shared_xaxes=True
)

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()

    rhl        = compute_rolling_half_life(spread).replace(np.inf, np.nan)
    rhl_smooth = rhl.ewm(span=21, adjust=False).mean()

    # Raw (faint background)
    fig.add_trace(go.Scatter(
        x=rhl.index, y=rhl.values,
        mode="lines",
        line=dict(width=0.8, color="steelblue"),
        opacity=0.3,
        name="Raw", showlegend=(i == 1)
    ), row=i, col=1)

    # EWM smoothed (prominent)
    fig.add_trace(go.Scatter(
        x=rhl_smooth.index, y=rhl_smooth.values,
        mode="lines",
        line=dict(width=1.8, color="steelblue"),
        name="EWM (21d)", showlegend=(i == 1)
    ), row=i, col=1)

    fig.add_hline(
        y=60, line=dict(color="red", dash="dash", width=1),
        annotation_text="60d threshold", annotation_position="top right",
        row=i, col=1
    )

fig.update_layout(
    height=300 * len(cointegrated_pairs),
    title_text="Rolling Half-Life (252-day window, EWM-smoothed) — Cointegrated Pairs",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.update_yaxes(title_text="Half-Life (days)", range=[0, 500])
fig.show()


In [67]:
n = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(5.85, 1.76 * n))
if n == 1:
    axes = [axes]
panel_labels = [f"({chr(97 + i)})" for i in range(n)]
for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    rhl        = compute_rolling_half_life(spread, window=252).replace(np.inf, np.nan)
    rhl_smooth = rhl.ewm(span=21, adjust=False).mean()
    pair_name  = f"{row['y']}/{row['x']}"
    ax.plot(rhl.index, rhl.values,
            linewidth=0.8, alpha=0.3,
            label="Raw" if label == panel_labels[0] else None)
    ax.plot(rhl_smooth.index, rhl_smooth.values,
            linewidth=1.8,
            label="EWM (21d)" if label == panel_labels[0] else None)
    ax.axhline(60, linestyle="--", linewidth=1, color="red")
    ax.text(rhl.index[-1], 62, r"60d threshold",
            ha="right", va="bottom", fontsize=10, color="red")
    ax.fill_between(
        rhl_smooth.index, rhl_smooth.values, 60,
        where=(rhl_smooth.fillna(0).values > 60),
        alpha=0.2, color="red"
    )
    ax.set_ylabel(r"Half-Life (days)")
    ax.set_ylim(0, 500)
    ax.text(0.02, 0.96, f"{label} {pair_name}",
            transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="top")
axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_,
           loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/01_spread_rolling_halflife.pdf", bbox_inches="tight")
plt.close(fig)


### Rolling z-score

In [68]:
n_pairs = len(cointegrated_pairs)
fig_plotly = make_subplots(
    rows=n_pairs, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r['y']}/{r['x']}" for _, r in cointegrated_pairs.iterrows()],
    vertical_spacing=0.08
)

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    spread = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    rz = compute_rolling_zscore(spread, window=60)

    fig_plotly.add_trace(
        go.Scatter(x=rz.index, y=rz.values, name=f"{row['y']}/{row['x']}",
                   line=dict(width=1), showlegend=False),
        row=i, col=1
    )
    for level, dash, label in [(2, "dot", "+2σ"), (-2, "dot", "-2σ"),
                                (1, "dash", "+1σ"), (-1, "dash", "-1σ"),
                                (0, "solid", "Mean")]:
        fig_plotly.add_hline(
            y=level, line=dict(color="gray", dash=dash, width=0.8),
            annotation_text=label if i == 1 else None,
            annotation_font_size=9,
            row=i, col=1
        )

fig_plotly.update_yaxes(title_text="Z-score", zeroline=True)
fig_plotly.update_xaxes(title_text="Date", row=n_pairs, col=1)
fig_plotly.update_layout(
    title="Rolling Z-Score (60-day window) — Cointegrated Pairs",
    height=350 * n_pairs,
    template="plotly_white"
)
fig_plotly.show()

In [69]:
n_pairs = len(cointegrated_pairs)
fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(5.9, 2.60 * n_pairs))
if n_pairs == 1:
    axes = [axes]
panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]
for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    spread = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    rz        = compute_rolling_zscore(spread, window=60)
    first     = (label == panel_labels[0])
    pair_name = f"{row['y']}/{row['x']}"
    line, = ax.plot(rz.index, rz.values, linewidth=1,
                    label="Z-score" if first else None)
    line_colour = line.get_color()
    ax.axhline( 2, linestyle=":",  linewidth=0.8,
                label=r"$\pm2\sigma$" if first else None)
    ax.axhline(-2, linestyle=":",  linewidth=0.8)
    ax.axhline( 1, linestyle="--", linewidth=0.8,
                label=r"$\pm1\sigma$" if first else None)
    ax.axhline(-1, linestyle="--", linewidth=0.8)
    ax.axhline( 0, linestyle="-",  linewidth=0.6, color="black",
                label="Mean" if first else None)
    ax.fill_between(rz.index,  2, rz.values.clip(min=2),   color=line_colour, alpha=0.15)
    ax.fill_between(rz.index, -2, rz.values.clip(max=-2),  color=line_colour, alpha=0.15)
    ax.set_title(f"{label} {pair_name}", loc="left", fontsize=10, fontweight="bold")
    ax.set_ylabel("Z-score")
    ax.annotate(r"$+2\sigma$", xy=(1,  2), xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$-2\sigma$", xy=(1, -2), xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$+1\sigma$", xy=(1,  1), xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$-1\sigma$", xy=(1, -1), xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
    ax.annotate(r"$\mu$",      xy=(1,  0), xycoords=("axes fraction", "data"), fontsize=8, va="center", ha="left", xytext=(3, 0), textcoords="offset points")
axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/01_spread_rolling_zscore.pdf", bbox_inches="tight")
plt.close(fig)


## Saving statistics data to CSV

In [ ]:
# Save confirmed cointegrated pairs with hedge ratios
cointegrated_pairs.to_csv("../../data/processed/cointegrated_pairs.csv", index=False)

# Save spread summary statistics
summary_df.to_csv("../../data/processed/spread_summary.csv")

# Save the log prices (needed for spread reconstruction in later notebooks)
prices_log.to_csv("../../data/processed/prices_log.csv")

## Conclusions

Of the 16 candidate pairs screened across eight sectors, **three pairs were confirmed as cointegrated** at the 5% significance level over the in-sample period (January 2018 -- December 2023):

| Pair | Hedge Ratio ($\hat{\beta}$) | AEG $p$-value | Half-Life (days) | Hurst Exponent | $\mu_S$ | $\sigma_S$ |
|---|---|---|---|---|---|---|
| GS.N / MS.N | 0.780 | 0.020 | 41.8 | 0.403 | 0.000 | 0.056 |
| KO.N / PEP.O | 0.641 | 0.035 | 40.3 | 0.421 | 0.000 | 0.047 |
| DAL.N / UAL.O | 0.672 | 0.036 | 52.9 | 0.414 | 0.000 | 0.076 |

**Cointegration results.** All log-price series were confirmed $I(1)$ --- non-stationary in levels, stationary in first differences --- satisfying the precondition for Engle--Granger testing. GS/MS and KO/PEP represent well-established fundamental linkages within Investment Banking and Beverages respectively. DAL/UAL captures the shared exposure of Delta and United Airlines to common drivers including fuel costs, load factors, and regulation. CVX/XOM was borderline ($p = 0.064$), while the remaining 12 pairs showed no evidence of a stable long-run equilibrium. Sector proximity alone is insufficient for cointegration; a common stochastic trend must genuinely link the two series.

**Spread characterisation.** All three spreads exhibit Hurst exponents below 0.5, confirming mean-reverting dynamics, and mean-reversion half-lives between 41 and 53 days. The spread means are numerically zero by OLS construction; standard deviations range from 0.047 to 0.076 in log-price units. These statistics are passed to `02_cointegration_validation.ipynb` to test whether the relationships persist out-of-sample.